# 0077: the read on test_real and test_dolly (arms R and L), and 0074's test and test_long (arm R), once

The answers only, free values, one decoder, each set read **once**: test_real (300 OASST1 prompts) and test_dolly
(300 Dolly-15k prompts, a source neither arm saw) for both arms; 0074's test and test_long for arm R (arm L's answers
there are 0074's read, `rules/reads/0074/view_long_l_test.jsonl`). This notebook refuses to run until dev's two
reports are committed in the code it clones, and it does not read again a file that is already whole. Nobody opens
these answers by eye: they are scored at home, counts only.

It writes `rules/reads/0077/view_real_{r,l}_test.jsonl` and `view_real_r_test0074.jsonl`.
**Disconnect when it prints `read: test`.**

In [ ]:
import hashlib, json, os, signal, subprocess, sys, time
BRANCH = 'master'
if not os.path.exists('/content/CubbyLLM'):
    !git clone -q -b {BRANCH} https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM
else:
    !cd /content/CubbyLLM && git fetch -q && git checkout -q {BRANCH} && git pull -q --ff-only
!pip -q install tokenizers safetensors numpy regex
from google.colab import drive; drive.mount('/content/drive')
REPO, DRIVE = '/content/CubbyLLM', '/content/drive/MyDrive/cubbyllm'
CKPT      = f'{DRIVE}/runs/base450m_cont/base450m_cont_final.pt'
TOKENIZER = f'{DRIVE}/token_cache_cont/tokenizer/grillcheese_bbpe128k.json'

PHASE = 'test'
ROWS     = f'{DRIVE}/rules/long_rows_0077/read_rows.jsonl'
ROWS74   = f'{DRIVE}/rules/long_rows_0074/read_rows.jsonl'
ADAPTERS = {'r': f'{DRIVE}/rules/cubby450m_view_real_r', 'l': f'{DRIVE}/rules/cubby450m_view_long_l'}
READS    = f'{DRIVE}/rules/reads/0077'
for p in [CKPT, TOKENIZER, ROWS, ROWS74] + [f'{d}/{x}' for d in ADAPTERS.values() for x in ('talk_lora.safetensors', 'talk_lora.json', 'DONE')]:
    assert os.path.exists(p), p
mans = {}
for rows_p, name in ((ROWS, '0077'), (ROWS74, '0074')):
    mans[name] = json.load(open(f'{REPO}/validation/logs/long_rows_{name}_read_rows.manifest.json'))
    assert hashlib.sha256(open(rows_p, 'rb').read()).hexdigest() == mans[name]['sha256'], f'{rows_p} is not the committed file'
    keys = {k for ln in open(rows_p, encoding='utf-8') for k in json.loads(ln)}
    assert keys == {'id', 'split', 'instruction'}, f'the rows file holds more than ids and instructions: {keys}'
    print(name, mans[name]['rows'], 'rows', mans[name]['by_split'])
os.makedirs(READS, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
for arm in 'rl':
    assert os.path.exists(f'{REPO}/validation/logs/view_real_dev_{arm}.json'), 'dev is read and scored first, and its reports committed'
PASSES = [(arm, ROWS, 'test_real,test_dolly', f'{READS}/view_real_{arm}_test.jsonl', mans['0077']) for arm in 'rl']
PASSES.append(('r', ROWS74, 'test,test_long', f'{READS}/view_real_r_test0074.jsonl', mans['0074']))

os.environ['CB_NO_FLEX'] = '1'
N = 300
for arm, rows_p, splits, OUT, man in PASSES:
    want = sum(min(N, man['by_split'][s]) for s in splits.split(','))
    have = [json.loads(ln) for ln in open(OUT, encoding='utf-8')] if os.path.exists(OUT) else []
    assert not any(d.get('own_text') for d in have), f'{OUT} was read with the own-text rule: move it aside'
    if len({d['id'] for d in have}) >= want:
        print(f'--- arm {arm}: {OUT} is whole ({len(have)} answers): not read again')
        continue
    print(f'--- arm {arm}: {splits} -> {OUT}')
    cmd = [sys.executable, '-u', 'validation/exp_view_split_torch.py', '--arm', 'm', '--ckpt', CKPT, '--tokenizer', TOKENIZER,
           '--adapter', ADAPTERS[arm], '--rows', rows_p, '--splits', splits, '--n', str(N), '--batch', '32', '--out', OUT]
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    assert p.wait() == 0, f'the read failed ({p.returncode})'
    !wc -l "{OUT}"
print('read:', PHASE, '(all passes read)')

## Done

**Disconnect the runtime** (on `read: test`). At home the answers are scored on the VM, counts only, and set against the order's lines; line 5 says which reader is kept.